In [ ]:
# 1. What is Memory?

# An LLM by itself is essentially:

# User
#  ↓
# Question
#  ↓
# LLM
#  ↓
# Answer




# Suppose:

# User: My name is Alok.

# AI: Nice to meet you!

# User: What is my name?




# If we send only:

# "What is my name?"

# the model has no guaranteed access to the previous turn.





# Memory changes the architecture:

# User
#  ↓
# Memory / History
#  ↓
# Previous messages + current message
#  ↓
# Gemini
#  ↓
# Response
#  ↓
# Save response




# The key idea:

# Memory is not the model remembering by itself. Your application stores previous state 
# and supplies the relevant state back to the model.

### Messages

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage

history = [
    HumanMessage(content="My name is Alok."),
    AIMessage(content="Nice to meet you, Alok!")
]

print(history[0].content)
print(history[1].content)



# So:

# HumanMessage = user
# AIMessage    = model
# SystemMessage = instructions
# ToolMessage   = tool result

My name is Alok.
Nice to meet you, Alok!


In [3]:
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite"
)

### Manual Conversation History

In [5]:
from langchain_core.messages import HumanMessage, AIMessage

history = []

def chat(user_input):
    history.append(HumanMessage(content=user_input))

    response = llm.invoke(history)

    history.append(AIMessage(content=response.text))

    return response.text

print(chat("My name is Alok")) 
print(chat("what is my name ?"))

Hello Alok! Nice to meet you. How can I help you today?
Your name is Alok.


In [ ]:
# First call

# history:
#     Human: My name is Alok

# Gemini
#     ↓

# history:
#     Human: My name is Alok
#     AI: Nice to meet you...





# Second call

# history:
#     Human: My name is Alok
#     AI: Nice to meet you...
#     Human: What is my name?

# Gemini
#     ↓

# AI: Your name is Alok.




# This is the foundation of every conversational memory system.

### Modern Short-Term Memory with Agent

In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model="google_genai:gemini-3.6-flash",
    tools=[],
    checkpointer=InMemorySaver()
)



# Now create a thread:
config = {
    "configurable": {
        "thread_id": "user-1"
    }
}

In [ ]:
# First Message
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "My name is Alok."
            }
        ]
    },
    config
)

print(result["messages"][-1].content)

In [ ]:
# Second message:

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "What is my name?"
            }
        ]
    },
    config
)

print(result["messages"][-1].content)

In [ ]:
# The same thread_id means:


# thread_id = user-1

# Message 1
#    ↓
# saved


# Message 2
#    ↓
# previous state loaded
#    ↓
# Gemini


# This is the modern replacement for the basic ConversationBufferMemory idea.

# The current docs use exactly this architecture: create_agent() + InMemorySaver() + thread_id.

### Why thread_id Matters

In [ ]:
# Suppose:


# thread_id = "user-1"

# has:
# My name is Alok.
# I study computer science.




# Another conversation:


# thread_id = "user-2"

# has:
# My name is Rahul.
# I am a designer.





# They remain separate.

#                  Agent
#                    │
#         ┌──────────┴──────────┐
#         │                     │
#    thread_id=user-1      thread_id=user-2
#         │                     │
#      Alok data             Rahul data



# This is extremely important for a real chatbot.